In [11]:
import os.path as osp

import gtdynamics as gtd
import gtsam
import numpy as np
import pandas as pd
import mujoco
import math

In [ ]:
URDF_PATH = "/home/havri/Programming/gtd/models/urdfs" # TODO: make the path relative


In [ ]:
URDF_PATH

In [12]:
robot = gtd.CreateRobotFromFile('../../models/urdfs/inverted_pendulum.urdf')

In [13]:
j1_id = robot.joint("j1").id()
robot = robot.fixLink("l1")

In [14]:
T = 3.0  # seconds
dt = 1. / 100  # Time horizon (s) and timestep duration (s).
t_steps = math.ceil(T / dt)  # Timesteps.

In [15]:
# Noise models:
dynamics_model = gtsam.noiseModel.Isotropic.Sigma(1, 1e-5)  # Dynamics constraints.
objectives_model = gtsam.noiseModel.Isotropic.Sigma(1, 1e-2)
control_model = gtsam.noiseModel.Isotropic.Sigma(1, 1e-1)

In [6]:
type(robot)

gtdynamics.gtdynamics.Robot

In [16]:
# Create trajectory factor graph.
gravity = (0, 0, -9.8)
planar_axis = (1, 0, 0)
graph_builder = gtd.DynamicsGraph(gravity, planar_axis)
graph = graph_builder.trajectoryFG(robot, t_steps, dt)

In [17]:
# Add initial conditions to trajectory factor graph.
theta_i = 0
dtheta_i = 0
graph.addPriorDouble(gtd.JointAngleKey(j1_id, 0), theta_i, dynamics_model)
graph.addPriorDouble(gtd.JointVelKey(j1_id, 0), dtheta_i, dynamics_model)

In [18]:
# Add state and min torque objectives to trajectory factor graph.
theta_T = np.pi
dtheta_T = 0
ddtheta_T = 0
graph.addPriorDouble(gtd.JointAngleKey(j1_id, t_steps), theta_T,
                        objectives_model)
graph.addPriorDouble(gtd.JointVelKey(j1_id, t_steps), dtheta_T,
                        objectives_model)
graph.addPriorDouble(gtd.JointAccelKey(j1_id, t_steps), ddtheta_T,
                        objectives_model)

In [19]:
# Apply torque costs at all steps.
for t in range(t_steps + 1):
    graph.add(gtd.MinTorqueFactor(gtd.TorqueKey(j1_id, t), control_model))

In [ ]:
# Initialize solution.
intializer = gtd.Initializer()
init_vals = intializer.ZeroValuesTrajectory(robot, t_steps, 0, 0.0, None)
print(init_vals.size())

3612


: 

In [11]:
# Optimize.
params = gtsam.LevenbergMarquardtParams()
params.setVerbosityLM("SUMMARY")
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, init_vals, params)
results = optimizer.optimize()

Initial error: 1.30091e+09, values: 3612
iter      cost      cost_change    lambda  success iter_time
   0  7.03775e+09     -5.7e+09      1e-05      1       0.03
iter      cost      cost_change    lambda  success iter_time
   0      5.1e+09     -3.8e+09     0.0001      1       0.03
iter      cost      cost_change    lambda  success iter_time
   0      2.4e+09     -1.1e+09      0.001      1       0.03
iter      cost      cost_change    lambda  success iter_time
   0      2.8e+09     -1.5e+09       0.01      1       0.03
iter      cost      cost_change    lambda  success iter_time
   0        5e+09     -3.7e+09        0.1      1       0.02
iter      cost      cost_change    lambda  success iter_time
   0      4.9e+09     -3.6e+09          1      1       0.03
iter      cost      cost_change    lambda  success iter_time
   0      6.6e+08      6.4e+08         10      1       0.02
   1      4.8e+08      1.8e+08          1      1       0.03
   2      3.6e+07      4.5e+08        0.1      1    

In [12]:
results

Values with 3612 values:
Value 18296968702853120: (Eigen::Matrix<double, 6, 1, 0, 6, 1>)
[
	5.2e-13;
	0;
	0;
	0;
	2.6e-12;
	-2.4e-16
]

Value 18296968702853121: (Eigen::Matrix<double, 6, 1, 0, 6, 1>)
[
	7.2e-13;
	0;
	0;
	0;
	3.6e-12;
	-7.2e-16
]

Value 18296968702853122: (Eigen::Matrix<double, 6, 1, 0, 6, 1>)
[
	6.6e-14;
	0;
	0;
	0;
	3.9e-13;
	-1.4e-16
]

Value 18296968702853123: (Eigen::Matrix<double, 6, 1, 0, 6, 1>)
[
	-5.9e-13;
	0;
	0;
	0;
	-2.9e-12;
	1.6e-15
]

Value 18296968702853124: (Eigen::Matrix<double, 6, 1, 0, 6, 1>)
[
	-1.2e-12;
	0;
	0;
	0;
	-6.1e-12;
	4e-15
]

Value 18296968702853125: (Eigen::Matrix<double, 6, 1, 0, 6, 1>)
[
	-1.9e-12;
	0;
	0;
	0;
	-9.3e-12;
	5.6e-15
]

Value 18296968702853126: (Eigen::Matrix<double, 6, 1, 0, 6, 1>)
[
	-2.5e-12;
	0;
	0;
	0;
	-1.3e-11;
	4e-15
]

Value 18296968702853127: (Eigen::Matrix<double, 6, 1, 0, 6, 1>)
[
	-3.1e-12;
	0;
	0;
	0;
	-1.6e-11;
	-4.3e-15
]

Value 18296968702853128: (Eigen::Matrix<double, 6, 1, 0, 6, 1>)
[
	-3.7e-12;
	0;
	0;


In [ ]:
model = mujoco.MjModel.from_xml_path('../../models/urdfs/inverted_pendulum.urdf')
data = mujoco.MjData(model)

In [8]:
model = mujoco.MjModel.from_xml_path('pendulum_scene.xml')
data = mujoco.MjData(model)

In [15]:
import mediapy as media

In [9]:
import mujoco
import mujoco.viewer

# Launch the viewer
mujoco.viewer.launch(model, data)

In [16]:
# Execute the torques 
print("Executing optimized torques...")
actuator_map = {model.joint(i).name: i for i in range(model.nu)}

mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

frames = []
with mujoco.Renderer(model, height=480, width=480) as renderer:
    for t in range(t_steps):
        # Apply the pre-computed torque for each joint
        for joint in robot.joints():
            if joint.name() in actuator_map:
                actuator_idx = actuator_map[joint.name()]
                torque = gtd.Torque(results, joint.id(), t)
                if t < 5: # Print for the first 5 steps
                    print(f"Step {t}, Joint: {joint.name()}, Torque: {torque}")
                data.ctrl[actuator_idx] = torque
        
        # Step the simulation forward for the duration of one planning step
        time_until = data.time + dt
        while data.time < time_until:
            mujoco.mj_step(model, data)

        # Render
        renderer.update_scene(data, camera="side_view")
        frames.append(renderer.render())

if frames:
    media.show_video(frames, fps=1/dt)

Executing optimized torques...
Step 0, Joint: j1, Torque: -0.0058632478116091786
Step 1, Joint: j1, Torque: -0.008111475893700407
Step 2, Joint: j1, Torque: -0.0008690519124683829


libdecor-gtk-WARNING: Failed to initialize GTK
Failed to load plugin 'libdecor-gtk.so': failed to init
No plugins found, falling back on no decorations


Step 3, Joint: j1, Torque: 0.006380324424669005
Step 4, Joint: j1, Torque: 0.013601309897954496


This browser does not support the video tag.
